# Radni tok s ljudskom kontrolom u Microsoft Agent Frameworku

## 🎯 Ciljevi učenja

U ovom bilježniku naučit ćete kako implementirati radne tokove s **ljudskom kontrolom** koristeći `RequestInfoExecutor` u Microsoft Agent Frameworku. Ovaj moćni obrazac omogućuje pauziranje AI radnih tokova kako bi se prikupili ljudski podaci, čineći agente interaktivnima i dajući ljudima kontrolu nad ključnim odlukama.

## 🔄 Što je ljudska kontrola (Human-in-the-Loop)?

**Ljudska kontrola (HITL)** je dizajnerski obrazac u kojem AI agenti pauziraju izvođenje kako bi zatražili ljudski unos prije nastavka. To je važno za:

- ✅ **Kritične odluke** - Dobivanje ljudskog odobrenja prije važnih radnji
- ✅ **Dvosmislene situacije** - Dopustiti ljudima da razjasne kada je AI nesiguran
- ✅ **Preferencije korisnika** - Pitati korisnike da odaberu između više opcija
- ✅ **Usklađenost i sigurnost** - Osigurati ljudski nadzor za regulirane operacije
- ✅ **Interaktivna iskustva** - Izgraditi konverzacijske agente koji odgovaraju na korisničke unose

## 🏗️ Kako to funkcionira u Microsoft Agent Frameworku

Framework pruža tri ključne komponente za HITL:

1. **`RequestInfoExecutor`** - Poseban izvršitelj koji pauzira tijek rada i emitira `RequestInfoEvent`
2. **`RequestInfoMessage`** - Osnovna klasa za tipizirane zahtjeve poslane ljudima
3. **`RequestResponse`** - Povezuje ljudske odgovore s izvornim zahtjevima koristeći `request_id`

**Obrazac radnog toka:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Naš primjer: Rezervacija hotela s potvrdom korisnika

Nadogradit ćemo uvjetni radni tok dodavanjem ljudske potvrde **prije** predlaganja alternativnih odredišta:

1. Korisnik traži odredište (npr. "Pariz")
2. `availability_agent` provjerava dostupnost soba
3. **Ako nema soba** → `confirmation_agent` pita "Želite li vidjeti alternative?"
4. Radni tok **pauzira** pomoću `RequestInfoExecutor`
5. **Čovjek odgovara** "da" ili "ne" putem konzole
6. `decision_manager` usmjerava na temelju odgovora:
   - **Da** → Prikaži alternativna odredišta
   - **Ne** → Otkaži zahtjev za rezervacijom
7. Prikaz završnog rezultata

Ovo demonstrira kako dati korisnicima kontrolu nad prijedlozima agenta!

---

Krenimo! 🚀


## Korak 1: Uvoz potrebnih knjižnica

Uvozimo standardne komponente Agent Frameworka plus **specifične klase za čovjeka u petlji**:
- `RequestInfoExecutor` - Izvršitelj koji pauzira tijek rada za unos čovjeka
- `RequestInfoEvent` - Događaj koji se emitira kada se traži unos čovjeka
- `RequestInfoMessage` - Osnovna klasa za tipizirane zahtjeve podataka
- `RequestResponse` - Povezuje odgovore čovjeka s zahtjevima
- `WorkflowOutputEvent` - Događaj za otkrivanje izlaza iz tijeka rada


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Korak 2: Definiranje Pydantic modela za strukturirane izlaze

Ovi modeli definiraju **shemu** koju će agenti vraćati. Zadržavamo sve modele iz uvjetnog tijeka rada i dodajemo:

**Novo za Human-in-the-Loop:**
- `HumanFeedbackRequest` - Podklasa `RequestInfoMessage` koja definira zahtjevni podatak poslan ljudima
  - Sadrži `prompt` (pitanje za postaviti) i `destination` (kontekst o nedostupnom gradu)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Korak 3: Izradite alat za rezervaciju hotela

Isti alat iz uvjetnog tijeka rada - provjerava jesu li sobe dostupne na odredištu.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Korak 4: Definirajte funkcije uvjeta za usmjeravanje

Trebamo **četiri funkcije uvjeta** za naš proces s uključenim čovjekom:

**Iz uvjetnog tijeka rada:**
1. `has_availability_condition` - Usmjerava kada su hoteli DOSTUPNI
2. `no_availability_condition` - Usmjerava kada hoteli NISU dostupni

**Novo za proces s uključenim čovjekom:**
3. `user_wants_alternatives_condition` - Usmjerava kada korisnik kaže "da" alternativama
4. `user_declines_alternatives_condition` - Usmjerava kada korisnik kaže "ne" alternativama


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Korak 5: Izrada izvršitelja Decision Managera

Ovo je **srce obrasca čovjeka-u-petlji**! `DecisionManager` je prilagođeni `Executor` koji:

1. **Prima povratne informacije od ljudi** putem objekata `RequestResponse`
2. **Procesira korisnikovu odluku** (da/ne)
3. **Usmjerava tijek rada** slanjem poruka odgovarajućim agentima

Ključne značajke:
- Koristi dekorator `@handler` za izlaganje metoda kao koraka tijeka rada
- Prima `RequestResponse[HumanFeedbackRequest, str]` koji sadrži i izvorni zahtjev i korisnikov odgovor
- Daje jednostavne poruke "da" ili "ne" koje pokreću naše funkcije uvjeta


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## Korak 6: Kreirajte prilagođeni izvršitelj prikaza

Isti izvršitelj prikaza iz uvjetnog tijeka rada - daje konačne rezultate kao izlaz tijeka rada.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Korak 7: Učitajte varijable okoline

Konfigurirajte LLM klijenta (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## Korak 8: Kreirajte AI agente i izvršitelje

Kreiramo **šest komponenti tijeka rada**:

**Agenti (umotani u AgentExecutor):**
1. **availability_agent** - Provjerava dostupnost hotela koristeći alat
2. **confirmation_agent** - 🆕 Priprema zahtjev za potvrdu od čovjeka
3. **alternative_agent** - Predlaže alternativne gradove (kad korisnik kaže da)
4. **booking_agent** - Potiče na rezervaciju (kad su sobe dostupne)
5. **cancellation_agent** - 🆕 Rukuje porukom o otkazivanju (kad korisnik kaže ne)

**Posebni izvršitelji:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` koji pauzira tijek rada za unos od čovjeka
7. **decision_manager** - 🆕 Prilagođeni izvršitelj koji usmjerava na temelju odgovora čovjeka (već definiran gore)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Korak 9: Izgradnja radnog toka s ljudskim sudjelovanjem

Sada gradimo graf radnog toka s **uvjetnim usmjeravanjem**, uključujući put s ljudskim sudjelovanjem:

**Struktura radnog toka:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Ključne veze:**
- `availability_agent → confirmation_agent` (kada nema soba)
- `confirmation_agent → prepare_human_request` (transformacija tipa)
- `prepare_human_request → request_info_executor` (pauza za čovjeka)
- `request_info_executor → decision_manager` (uvijek - pruža RequestResponse)
- `decision_manager → alternative_agent` (kada korisnik kaže "da")
- `decision_manager → cancellation_agent` (kada korisnik kaže "ne")
- `availability_agent → booking_agent` (kada su sobe dostupne)
- Svi putevi završavaju na `display_result`


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Korak 10: Pokreni testni slučaj 1 - Grad BEZ dostupnosti (Pariz s potvrdom čovjeka)

Ovaj test prikazuje **cijeli ciklus s uključenim čovjekom**:

1. Zahtjev za hotelom u Parizu
2. availability_agent provjerava → Nema soba
3. confirmation_agent kreira pitanje za čovjeka
4. request_info_executor **pauzira tijek rada** i šalje `RequestInfoEvent`
5. **Aplikacija otkriva događaj i pita korisnika u konzoli**
6. Korisnik upisuje "da" ili "ne"
7. Aplikacija šalje odgovor putem `send_responses_streaming()`
8. decision_manager usmjerava na temelju odgovora
9. Prikazan konačni rezultat

**Ključni uzorak:**
- Koristite `workflow.run_stream()` za prvu iteraciju
- Koristite `workflow.send_responses_streaming(pending_responses)` za kasnije iteracije
- Slušajte `RequestInfoEvent` da otkrijete kada je potreban unos čovjeka
- Slušajte `WorkflowOutputEvent` za hvatanje konačnih rezultata


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Korak 11: Pokreni Testni Slučaj 2 - Grad S Dostatnošću (Stockholm - Nije Potreban Ljudski Unos)

Ovaj test prikazuje **izravni put** kada su sobe dostupne:

1. Zahtjev za hotel u Stockholmu
2. availability_agent provjerava → Sobe dostupne ✅
3. booking_agent predlaže rezervaciju
4. display_result prikazuje potvrdu
5. **Nije potreban ljudski unos!**

Radni tijek u potpunosti zaobilazi ljudski element kada su sobe dostupne.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Ključni Zaključci i Najbolje Prakse za Human-in-the-Loop

### ✅ Što ste naučili:

#### 1. **RequestInfoExecutor Uzorak**
Uzorak human-in-the-loop u Microsoft Agent Framework koristi tri ključne komponente:
- `RequestInfoExecutor` - Pauzira tijek rada i emitira događaje
- `RequestInfoMessage` - Osnovna klasa za tipizirane zahtjeve (podklasirajte ju!)
- `RequestResponse` - Korelira ljudske odgovore s izvornim zahtjevima

**Ključno razumijevanje:**
- `RequestInfoExecutor` NE prikuplja unos samostalno - samo pauzira tijek rada
- Vaš aplikacijski kod mora slušati `RequestInfoEvent` i prikupljati unos
- Morate pozvati `send_responses_streaming()` s rječnikom koji mapira `request_id` na odgovor korisnika

#### 2. **Streaming Uzorak Izvršavanja**
```python
# Prva iteracija
stream = workflow.run_stream(initial_request)

# Naknadne iteracije (nakon unosa čovjeka)
stream = workflow.send_responses_streaming(pending_responses)

# Uvijek obrađuj događaje
events = [event async for event in stream]
```

#### 3. **Arhitektura Pokretana Događajima**
Slušajte specifične događaje za upravljanje tijekom rada:
- `RequestInfoEvent` - Potreban je ljudski unos (tijek rada pauziran)
- `WorkflowOutputEvent` - Konačni rezultat je dostupan (tijek rada završen)
- `WorkflowStatusEvent` - Promjene stanja (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, itd.)

#### 4. **Prilagođeni Izvršitelji s @handler**
`DecisionManager` pokazuje kako kreirati izvršitelje koji:
- Koriste dekorator `@handler` za izlaganje metoda kao koraka tijeka rada
- Primaju tipizirane poruke (npr. `RequestResponse[HumanFeedbackRequest, str]`)
- Usmjeravaju tijek rada slanjem poruka drugim izvršiteljima
- Pristupaju kontekstu preko `WorkflowContext`

#### 5. **Uvjetno Usmjeravanje s Ljudskim Odlukama**
Možete kreirati funkcije uvjeta koje evaluiraju ljudske odgovore:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Primjene u Stvarnom Svijetu:

1. **Radni Tijekovi Odobrenja**
   - Dobiti odobrenje menadžera prije obrade troškovnih izvještaja
   - Potrebna ljudska provjera prije slanja automatiziranih emailova
   - Potvrditi transakcije visoke vrijednosti prije izvršenja

2. **Moderacija Sadržaja**
   - Označiti sumnjivi sadržaj za ljudsku provjeru
   - Zamoliti moderatore da donesu konačnu odluku kod složenih slučajeva
   - Eskalirati ljudima kada je AI povjerenje nisko

3. **Korisnička Podrška**
   - Dopustiti AI-u da automatski odgovara na rutinska pitanja
   - Eskalirati složena pitanja ljudskim agentima
   - Pitati korisnika želi li razgovarati s ljudskim agentom

4. **Obrada Podataka**
   - Zamoliti ljude da riješe nejasne unose podataka
   - Potvrditi AI interpretacije nejasnih dokumenata
   - Dopustiti korisnicima da biraju između više valjanih interpretacija

5. **Sustavi Kritični za Sigurnost**
   - Zahtijevati ljudsku potvrdu prije nepovratnih akcija
   - Dobiti odobrenje prije pristupa osjetljivim podacima
   - Potvrditi odluke u reguliranim industrijama (zdravstvo, financije)

6. **Interaktivni Agenti**
   - Izgraditi konverzacijske botove koji postavljaju dodatna pitanja
   - Kreirati čarobnjake koji vode korisnike kroz složene procese
   - Dizajnirati agente koji surađuju s ljudima korak po korak

### 🔄 Usporedba: Sa i Bez Human-in-the-Loop

| Značajka | Uvjetni Tijek Rada | Human-in-the-Loop Tijek Rada |
|---------|---------------------|---------------------------|
| **Izvršavanje** | Jedan `workflow.run()` | Petlja s `run_stream()` + `send_responses_streaming()` |
| **Unos Korisnika** | Nema (potpuno automatizirano) | Interaktivni upiti putem `input()` ili UI |
| **Komponente** | Agent + Izvršitelji | + RequestInfoExecutor + DecisionManager |
| **Događaji** | Samo AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, itd. |
| **Pauziranje** | Nema pauziranja | Tijek rada pauzira na RequestInfoExecutor |
| **Ljudska Kontrola** | Nema ljudske kontrole | Ljudi donose ključne odluke |
| **Primjena** | Automatizacija | Suradnja i nadzor |

### 🚀 Napredni Uzorci:

#### Više Točaka Ljudskih Odluka
Možete imati više `RequestInfoExecutor` čvorova u istom tijeku rada:
```python
.add_edge(agent1, request_info_1)  # Prva ljudska odluka
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Druga ljudska odluka
.add_edge(decision_manager_2, final_agent)
```

#### Rukovanje Timeoutom
Implementirajte tajmere za ljudske odgovore:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Zadano na sigurnu opciju
```

#### Bogata Integracija UI-a
Umjesto `input()`, integrirajte se s web UI, Slack, Teams, itd.:
```python
if isinstance(event, RequestInfoEvent):
    # Pošalji obavijest na korisnikov željeni kanal
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Uvjetni Human-in-the-Loop
Ljudski unos tražite samo u specifičnim situacijama:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Preusmjeri na čovjeka samo ako je povjerenje nisko ili je vrijednost visoka
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Najbolje Prakse:

1. **Uvijek Podklasirajte RequestInfoMessage**
   - Pruža tipnu sigurnost i validaciju
   - Omogućuje bogat kontekst za prikaz UI-a
   - Jasnoća namjere svakog tipa zahtjeva

2. **Koristite Opisne Upite**
   - Uključite kontekst o tome što tražite
   - Objasnite posljedice svakog izbora
   - Držite pitanja jednostavnima i jasnima

3. **Rukujte Neočekivanim Unosom**
   - Validirajte korisničke odgovore
   - Pružite zadane vrijednosti za nevažeći unos
   - Dajte jasne poruke o greškama

4. **Pratite ID-ove Zahtjeva**
   - Koristite korelaciju između request_id i odgovora
   - Nemojte pokušavati ručno upravljati stanjem

5. **Dizajnirajte za Ne-Blocking**
   - Ne blokirajte niti čekajući unos
   - Koristite asinhrone obrasce kroz cijeli sustav
   - Podržavajte konkurentne instance tijeka rada

### 📚 Povezani Koncepti:

- **Agent Middleware** - Presretanje poziva agenta (prijašnji bilježnik)
- **Upravljanje Stanjima Tijeka Rada** - Čuvanje stanja tijeka rada između pokretanja
- **Suradnja Višestrukih Agenata** - Kombinacija human-in-the-loop s timovima agenata
- **Arhitekture Pokretane Događajima** - Izgradnja reaktivnih sustava s događajima

---

### 🎓 Čestitke!

Savladali ste tijekove rada human-in-the-loop s Microsoft Agent Frameworkom! Sada znate kako:
- ✅ Pauzirati tijekove rada za prikupljanje ljudskog unosa
- ✅ Koristiti RequestInfoExecutor i RequestInfoMessage
- ✅ Rukovati streaming izvršavanjem s događajima
- ✅ Kreirati prilagođene izvršitelje s @handler
- ✅ Usmjeravati tijekove rada na temelju ljudskih odluka
- ✅ Izgraditi interaktivne AI agente koji surađuju s ljudima

**Ovo je ključni uzorak za izgradnju pouzdanih, kontroliranih AI sustava!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
